In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("\n--- NGÀY 69: TỪ ĐIỂN SỐ HỌC (TOKENIZATION & PADDING) ---")

#1.Giả lập đánh giá của KH
danh_gia_cua_kh = [
    "Sản phẩm này rất tốt",
    "Rất tệ, tôi không thích sản phẩm này",
    "Tốt",
    "Giao hàng nhanh, sản phẩm đẹp xuất sắc"
]

#2.Khởi tạo người lập từ điển (Tokenizer)
#num_row=100 chỉ lưu lại 100 từ phổ biến nhất
#ovv_token="LẠ" thay thế các từ không có trong từ điển là LẠ
may_dich = Tokenizer(num_words=100, oov_token="<LẠ>")
#cho máy đọc qua dữ liệu, để tạo ra từ điển từ nào xuất hiện nhiều nhất
may_dich.fit_on_texts(danh_gia_cua_kh)

#In cuốn từ điển
tu_dien =may_dich.word_index
print(f"Từ điển AI vừa lập được (Tổng cộng {len(tu_dien)} từ):")
print(tu_dien)

#3.Chuyển chữ thành chuỗi số
chuoi_so = may_dich.texts_to_sequences(danh_gia_cua_kh)
print("\n Các câu văn sau khi chuyển thành chuỗi số")

for i in range(len(danh_gia_cua_kh)):
    print(f"[{danh_gia_cua_kh[i]}] ---> {chuoi_so[i]}")
    

#4.Kéo dãn/cắt gọt cho bằng nhau (Padding)
du_lieu_chuan = pad_sequences(chuoi_so, maxlen=7, padding='pre')

print("\n--- MA TRẬN CUỐI CÙNG ĐỂ ĐƯA VÀO MẠNG NƠ-RON ---")
print(du_lieu_chuan)


--- NGÀY 69: TỪ ĐIỂN SỐ HỌC (TOKENIZATION & PADDING) ---
Từ điển AI vừa lập được (Tổng cộng 16 từ):
{'<LẠ>': 1, 'sản': 2, 'phẩm': 3, 'này': 4, 'rất': 5, 'tốt': 6, 'tệ': 7, 'tôi': 8, 'không': 9, 'thích': 10, 'giao': 11, 'hàng': 12, 'nhanh': 13, 'đẹp': 14, 'xuất': 15, 'sắc': 16}

 Các câu văn sau khi chuyển thành chuỗi số
[Sản phẩm này rất tốt] ---> [2, 3, 4, 5, 6]
[Rất tệ, tôi không thích sản phẩm này] ---> [5, 7, 8, 9, 10, 2, 3, 4]
[Tốt] ---> [6]
[Giao hàng nhanh, sản phẩm đẹp xuất sắc] ---> [11, 12, 13, 2, 3, 14, 15, 16]

--- MA TRẬN CUỐI CÙNG ĐỂ ĐƯA VÀO MẠNG NƠ-RON ---
[[ 0  0  2  3  4  5  6]
 [ 7  8  9 10  2  3  4]
 [ 0  0  0  0  0  0  6]
 [12 13  2  3 14 15 16]]


In [8]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D

print("\n--- NGÀY 70: HUẤN LUYỆN AI PHÂN TÍCH CẢM XÚC (SENTIMENT ANALYSIS) ---")

# 1. Dán nhãn cho dữ liệu (1: Khen / Tích cực, 0: Chê / Tiêu cực)
# Câu 1: "Sản phẩm này rất tốt" -> 1
# Câu 2: "Rất tệ..." -> 0
# Câu 3: "Tốt" -> 1
# Câu 4: "Giao hàng nhanh..." -> 1
nhan_cam_xuc = np.array([1, 0, 1, 1])

# 2. Xây dựng bộ não Đọc hiểu Ngôn ngữ
model_nlp = Sequential()

# Lớp ĐẦU TIÊN vĩ đại nhất của NLP: Lớp Nhúng từ (Embedding)
# input_dim=100: Kích thước cuốn từ điển (ở Ngày 69 ta định nghĩa max 100 từ)
# output_dim=16: Ép mỗi từ thành một tọa độ vector 16 chiều (Bản đồ 16D)
# input_length=7: Vì chiều dài các câu (đã padding) đều là 7
model_nlp.add(Embedding(input_dim=100, output_dim=16, input_length=7))

# Lớp GlobalAveragePooling1D: Lấy trung bình cộng ý nghĩa của cả 7 từ trong câu
# Giúp AI tóm tắt đại ý của toàn bộ câu văn
model_nlp.add(GlobalAveragePooling1D())

# Lớp Ẩn: 16 nơ-ron suy nghĩ
model_nlp.add(Dense(16, activation='relu'))

# Lớp Đầu ra: 1 nơ-ron (Dùng Sigmoid để chốt hạ Khen=1 hay Chê=0)
model_nlp.add(Dense(1, activation='sigmoid'))

# 3. Biên dịch mô hình
model_nlp.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 4. Ép AI đi học (Với dữ liệu nhỏ tí xíu này, cho nó cày 50 vòng)
print("Bắt đầu huấn luyện khả năng đọc hiểu...")
model_nlp.fit(du_lieu_chuan, nhan_cam_xuc, epochs=50, verbose=0)
print("Huấn luyện xong!")

# 5. THỬ THÁCH AI VỚI MỘT CÂU VĂN MỚI TOANH
cau_moi = ["Sản phẩm rất tệ"]
print(f"\nKhách hàng mới bình luận: '{cau_moi[0]}'")

# Tiền xử lý câu mới y hệt quy trình Ngày 69
so_hoa_cau_moi = may_dich.texts_to_sequences(cau_moi)
cau_moi_chuan = pad_sequences(so_hoa_cau_moi, maxlen=7, padding='pre')

# Bắt AI dự đoán
phan_quyet = model_nlp.predict(cau_moi_chuan)

if phan_quyet[0][0] > 0.5:
    print(f"=> AI phán: Đây là lời KHEN (Tích cực) - Độ tự tin: {phan_quyet[0][0]*100:.2f}%")
else:
    print(f"=> AI phán: Đây là lời CHÊ (Tiêu cực) - Độ tự tin: {(1 - phan_quyet[0][0])*100:.2f}%")


--- NGÀY 70: HUẤN LUYỆN AI PHÂN TÍCH CẢM XÚC (SENTIMENT ANALYSIS) ---


c:\Users\ACER\anaconda3\envs\jarvis_ai\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Bắt đầu huấn luyện khả năng đọc hiểu...
Huấn luyện xong!

Khách hàng mới bình luận: 'Sản phẩm rất tệ'
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
=> AI phán: Đây là lời KHEN (Tích cực) - Độ tự tin: 63.31%
